# `standings.py` Reference

Accumulates per-player `Record` objects from a `Tournament`.
Groups and ranks them for use by pairing algorithms.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

ready


In [3]:
from tournament.generators import make_players, skilled_match, random_match
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing

rng = Random(42)
players = make_players(20, rng)
tour = run_tournament(players, n_rounds=6, pairing=get_pairing('adjacent'), rng=rng)

## `Record` — properties

In [14]:
from tournament.standings import compute_records

records = compute_records(tour)
print(f'  {"Name":<8} {"Skill":<8} {"W":>3} {"L":>3} {"diff":>6} {"total_ratio":>12}') 
print('  ' + '-'*66)
players = [player for player in tour.players]
players.sort(key=lambda p: p.skill, reverse=True)
for player in players:
    r = records[player.pid]
    print(f'  {player.name:<8} {r.skill:<8.3f} {r.wins:>3} {r.losses:>3} {r.agent_diff:>+6} '
          f'{r.agent_total_ratio:>12.3f}')

  Name     Skill      W   L   diff  total_ratio
  ------------------------------------------------------------------
  P017     1.311      8   4    +12        0.625
  P011     1.164     10   2    +21        0.714
  P003     0.702      8   4    +12        0.636
  P012     0.657      9   3     +6        0.562
  P006     0.332      7   5     +6        0.568
  P016     0.246      6   6     -3        0.469
  P010     0.232      7   5     -4        0.458
  P009     0.116      6   6     +0        0.500
  P013     0.111      7   5     +2        0.520
  P018     0.042      6   6     +4        0.537
  P019     -0.106     5   7     -7        0.434
  P002     -0.111     7   5     +4        0.545
  P004     -0.128     5   7     -4        0.460
  P000     -0.144     5   7     -3        0.471
  P001     -0.173     6   6     +1        0.510
  P008     -0.217     5   7     -3        0.469
  P007     -0.267     4   8     -6        0.435
  P014     -0.738     5   7     -2        0.480
  P015     -1.015  

## `Record` — player link

When a `Record` is built from a live `Tournament` (via `compute_records`), it
stores the source `Player` on `record.player`. Convenience accessors expose the
player's `name` and `skill` directly from the record.

If no player is attached, both properties return `None`.

In [15]:
r = records[players[0].pid]

print('Record from compute_records:')
print(f'  pid    = {r.pid}')
print(f'  name   = {r.name}')
print(f'  skill  = {r.skill}')
print(f'  player = {r.player}')

print()

from tournament.standings import Record
bare = Record()
print('Bare Record (no player):')
print(f'  name   = {bare.name}')
print(f'  skill  = {bare.skill}')
print(f'  pid    = {bare.pid}')

Record from compute_records:
  pid    = 17
  name   = P017
  skill  = 1.3110808272040482
  player = Player(pid=17, name='P017', skill=1.3110808272040482)

Bare Record (no player):
  name   = None
  skill  = None
  pid    = -1


## `Record.agent_seq` — raw agent sequence

In [19]:
def compute_ratio(seq:list[tuple[int, int]]):
    f = sum(f for f, a in seq)
    a = sum(a for f, a in seq)
    return f/(f+a)

pid = players[0].pid
r = records[pid]
print(f'{players[pid].name} agent sequence (agents_for, agents_against) per round:')
running_diff = 0
previous = []

for i, (f, a) in enumerate(r.agent_seq, 1):
    running_diff += f - a
    previous.append((f, a))
    print(f'  round {i}: for={f}  against={a}  diff={running_diff:+d}  ratio={compute_ratio(previous):.2f}')

P014 agent sequence (agents_for, agents_against) per round:
  round 1: for=5  against=3  diff=+2  ratio=0.62
  round 2: for=6  against=2  diff=+6  ratio=0.69
  round 3: for=6  against=1  diff=+11  ratio=0.74
  round 4: for=5  against=6  diff=+10  ratio=0.65
  round 5: for=6  against=0  diff=+16  ratio=0.70
  round 6: for=2  against=6  diff=+12  ratio=0.62


## `compute_records` — `through_round` snapshot

In [7]:
for r_num in range(1, len(tour.rounds) + 1):
    snap = compute_records(tour, through_round=r_num)
    wins = [(players[pid].name, rec.wins) for pid, rec in snap.items()]
    wins.sort(key=lambda x: x[1], reverse=True)
    print(f'After round {r_num}: {wins}')

After round 1: [('P003', 2), ('P004', 2), ('P006', 2), ('P011', 2), ('P012', 2), ('P014', 2), ('P000', 1), ('P001', 1), ('P008', 1), ('P009', 1), ('P016', 1), ('P017', 1), ('P018', 1), ('P019', 1), ('P002', 0), ('P005', 0), ('P007', 0), ('P010', 0), ('P013', 0), ('P015', 0)]
After round 2: [('P011', 4), ('P012', 4), ('P001', 3), ('P003', 3), ('P006', 3), ('P017', 3), ('P004', 2), ('P008', 2), ('P009', 2), ('P010', 2), ('P013', 2), ('P014', 2), ('P018', 2), ('P019', 2), ('P000', 1), ('P002', 1), ('P007', 1), ('P016', 1), ('P005', 0), ('P015', 0)]
After round 3: [('P011', 6), ('P017', 5), ('P001', 4), ('P006', 4), ('P010', 4), ('P012', 4), ('P013', 4), ('P000', 3), ('P003', 3), ('P004', 3), ('P008', 3), ('P014', 3), ('P018', 3), ('P002', 2), ('P009', 2), ('P015', 2), ('P016', 2), ('P019', 2), ('P007', 1), ('P005', 0)]
After round 4: [('P011', 7), ('P010', 6), ('P017', 6), ('P000', 5), ('P006', 5), ('P012', 5), ('P013', 5), ('P001', 4), ('P002', 4), ('P003', 4), ('P004', 4), ('P008', 4), 

## `group_by_record` + `sort_groups`

In [20]:
from tournament.standings import group_by_record, sort_groups

groups   = group_by_record(records)
sorted_g = sort_groups(groups)   # default tiebreaker: agent_differential

print('Groups (sorted by agent_differential within each record):')
for label, recs in sorted_g.items():
    items = [(players[r.pid].name, r.agent_diff, r.skill) for r in recs]
    print(f'  {label}: {items}')

Groups (sorted by agent_differential within each record):
  10-2: [('P002', 21, 1.163558686599143)]
  9-3: [('P004', 6, 0.6566365067986689)]
  8-4: [('P012', 12, 0.7019837250988631), ('P014', 12, 1.3110808272040482)]
  7-5: [('P010', 6, 0.33231834406771527), ('P003', 4, -0.11131586156766246), ('P000', 2, 0.11050717744383194), ('P019', -4, 0.23229773690672087)]
  6-6: [('P015', 4, 0.04165686390338389), ('P011', 1, -0.1729036003315193), ('P018', 0, 0.11588478670085507), ('P007', -3, 0.24634219521120196)]
  5-7: [('P001', -2, -0.7383216023448206), ('P017', -3, -0.14409032957792836), ('P013', -3, -0.216958684145195), ('P006', -4, -0.12758828378288709), ('P005', -7, -0.10632329377078427)]
  4-8: [('P009', -6, -0.2673374784971682)]
  3-9: [('P008', -15, -1.014662367487717)]
  1-11: [('P016', -21, -1.4973534143409575)]


## `make_groups_even` / `create_ranked_even_groups`

`make_groups_even` balances odd-sized record brackets by moving one player to the
next bracket, so every bracket has an even number of players. `create_ranked_even_groups`
chains the full pipeline: `group_by_record` → `sort_groups` → `make_groups_even`.

This is the default input used by `pairing.make_record_group_pairing`.

The key fix is that groups are processed **best-record-first** (numeric order, not
lexicographic string order), so the carried player always moves from a better
bracket to the next worse bracket.

In [21]:
from tournament.standings import make_groups_even, create_ranked_even_groups

# Make raw record groups even-sized
raw_groups = group_by_record(records)
print('Raw groups:')
for label, recs in raw_groups.items():
    items = [(players[r.pid].name, r.wins, r.losses, r.agent_diff) for r in recs]
    print(f'  {label}: {items}')
sorted_groups = sort_groups(raw_groups)
print('Sorted groups:')
for label, recs in sorted_groups.items():
    items = [(players[r.pid].name, r.wins, r.losses, r.agent_diff) for r in recs]
    print(f'  {label}: {items}')
even_groups = make_groups_even(sorted_groups)

print('Even-sized groups (keys have ~ suffix):')
for label, recs in even_groups.items():
    items = [(players[r.pid].name, r.wins, r.losses, r.agent_diff) for r in recs]
    print(f'  {label}: {items}')

print()

# One-call convenience
print('create_ranked_even_groups:')
for label, recs in create_ranked_even_groups(records).items():
    print(f'  {label}: {len(recs)} players')

Raw groups:
  10-2: [('P002', 10, 2, 21)]
  9-3: [('P004', 9, 3, 6)]
  8-4: [('P012', 8, 4, 12), ('P014', 8, 4, 12)]
  7-5: [('P003', 7, 5, 4), ('P010', 7, 5, 6), ('P019', 7, 5, -4), ('P000', 7, 5, 2)]
  6-6: [('P011', 6, 6, 1), ('P018', 6, 6, 0), ('P007', 6, 6, -3), ('P015', 6, 6, 4)]
  5-7: [('P017', 5, 7, -3), ('P006', 5, 7, -4), ('P013', 5, 7, -3), ('P001', 5, 7, -2), ('P005', 5, 7, -7)]
  4-8: [('P009', 4, 8, -6)]
  3-9: [('P008', 3, 9, -15)]
  1-11: [('P016', 1, 11, -21)]
Sorted groups:
  10-2: [('P002', 10, 2, 21)]
  9-3: [('P004', 9, 3, 6)]
  8-4: [('P012', 8, 4, 12), ('P014', 8, 4, 12)]
  7-5: [('P010', 7, 5, 6), ('P003', 7, 5, 4), ('P000', 7, 5, 2), ('P019', 7, 5, -4)]
  6-6: [('P015', 6, 6, 4), ('P011', 6, 6, 1), ('P018', 6, 6, 0), ('P007', 6, 6, -3)]
  5-7: [('P001', 5, 7, -2), ('P017', 5, 7, -3), ('P013', 5, 7, -3), ('P006', 5, 7, -4), ('P005', 5, 7, -7)]
  4-8: [('P009', 4, 8, -6)]
  3-9: [('P008', 3, 9, -15)]
  1-11: [('P016', 1, 11, -21)]
Even-sized groups (keys have ~ 

## `make_rank_key` — custom tiebreaker

In [10]:
from tournament.standings import make_rank_key

# Default key: agent_differential
key_default = make_rank_key()

# Custom: total agents scored only
key_scored = make_rank_key(rating_fn=lambda seq: sum(f for f, _ in seq))

print(f'  {"Name":<8} {"W":>3} {"default_key":>12} {"scored_key":>12}')
print('  ' + '-'*38)
for r in sorted(records.values(), key=key_default, reverse=True):
    print(f'  {players[r.pid].name:<8} {r.wins:>3} '
          f'{key_default(r):>12.3f} {key_scored(r):>12.3f}')

  Name       W  default_key   scored_key
  --------------------------------------
  P011      10       21.000       35.000
  P003       8       12.000       28.000
  P017       8       12.000       30.000
  P006       7        6.000       25.000
  P012       9        6.000       27.000
  P002       7        4.000       24.000
  P018       6        4.000       29.000
  P013       7        2.000       26.000
  P001       6        1.000       26.000
  P009       6        0.000       23.000
  P014       5       -2.000       24.000
  P000       5       -3.000       24.000
  P008       5       -3.000       23.000
  P016       6       -3.000       23.000
  P004       5       -4.000       23.000
  P010       7       -4.000       22.000
  P007       4       -6.000       20.000
  P019       5       -7.000       23.000
  P015       3      -15.000       17.000
  P005       1      -21.000       13.000


## `agent_differential` — standalone

In [11]:
from tournament.standings import agent_differential

seq = records[players[0].pid].agent_seq
print(f'agent_seq:           {seq}')
print(f'agent_differential:  {agent_differential(seq)}')

agent_seq:           [(5, 3), (2, 6), (6, 4), (6, 2), (2, 6), (3, 6)]
agent_differential:  -3
